In [1]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
#from ucimlrepo import fetch_ucirepo 
import pandas as pd
import random

In [12]:
# fetch dataset 
#wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
#x = wine_quality.data.features 
#y = wine_quality.data.targets 

#df = pd.concat([x,y], axis=1)
#print(df.head())
# metadata 
#print(wine_quality.metadata) 
  
# variable information 
#print(wine_quality.variables) 

#print(len(wine_quality.data.features))
#print(df.columns)


#dataset terpisah (white wine)
df = pd.read_csv(
    "winequality-white-indonesia.csv",
    sep=";",
    decimal=","
)

print(df.columns)
print(len(df))

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')
4898


In [40]:
idx = random.randint(0,4897)

fa = df["fixed acidity"][idx]
rs = df["residual sugar"][idx]
d = df["density"][idx]
p = df["pH"][idx]
s = df["sulphates"][idx]
a = df["alcohol"][idx]

print(fa, rs, d, p, s, a, idx)

6.9 1.15 0.99047 3.11 0.38 11.4 4030


In [41]:
def perhitungan_triangle(x, batas_kiri, tengah, batas_kanan):
    if x <= batas_kiri or x >= batas_kanan:
        return 0
    elif x == tengah:
        return 1
    elif x < tengah:
        return (x-batas_kiri)/(tengah-batas_kiri)
    else:
        return (batas_kanan-x)/(batas_kanan - tengah)

In [42]:
def perhitungan_trapezoidal(x, a, b, c, d):
    if x <= a or x >= d:
        return 0
    elif x > a and x < b:
        return (x-a)/(b-a)
    elif x >= b and x <= c:
        return 1
    else:
        return (d-x)/(d-c)

In [43]:
def perbandingan(tinggi, sedang, rendah):
    if rendah >= sedang and rendah >= tinggi:
        return "rendah"

    elif sedang >= tinggi:
        return "sedang"

    else:
        return "tinggi"

In [44]:
def nilai_fa(x):
    tinggi = perhitungan_trapezoidal(x,8,10,15,15)
    sedang = perhitungan_trapezoidal(x,5,6,8,9)
    rendah = perhitungan_trapezoidal(x,3,3,5,6)

    dict_fa = {
        "tinggi" : tinggi,
        "sedang" : sedang,
        "rendah" : rendah
    }
    
    #print(tinggi, sedang, rendah)
    #hasil = perbandingan(tinggi, sedang, rendah)
    return dict_fa

In [45]:
def nilai_rs(x):
    tinggi = perhitungan_trapezoidal(x,15,20,70,70)
    sedang = perhitungan_trapezoidal(x,4,5,15,20)
    rendah = perhitungan_trapezoidal(x,0,0,4,5)

    dict_rs = {
        "tinggi" : tinggi,
        "sedang" : sedang,
        "rendah" : rendah
    }
    
    return dict_rs

In [46]:
def nilai_den(x):
    tinggi = perhitungan_trapezoidal(x,0.998,1,1.04,1.04)
    normal = perhitungan_trapezoidal(x,0.992,0.994,0.998,1)
    rendah = perhitungan_trapezoidal(x,0.987,0.987,0.992,0.994)

    dict_den = {
        "tinggi" : tinggi,
        "normal" : normal,
        "rendah" : rendah
    }
    
    return dict_den

In [47]:
def nilai_ph(x):
    tinggi = perhitungan_trapezoidal(x,3.4,3.5,4,4)
    normal = perhitungan_trapezoidal(x,3,3.1,3.4,3.5)
    rendah = perhitungan_trapezoidal(x,2.7,2.7,3,3.1)

    dict_ph = {
        "basa" : tinggi,
        "normal" : normal,
        "asam" : rendah
    }
    
    return dict_ph

In [48]:
def nilai_sul(x):
    tinggi = perhitungan_trapezoidal(x,0.7,0.8,1.2,1.2)
    normal = perhitungan_trapezoidal(x,0.4,0.5,0.7,0.8)
    rendah = perhitungan_trapezoidal(x,0.2,0.2,0.4,0.5)

    dict_sul = {
        "tinggi" : tinggi,
        "sedang" : normal,
        "rendah" : rendah
    }
    
    return dict_sul

In [49]:
def nilai_al(x):
    tinggi = perhitungan_trapezoidal(x,12,13,15,15)
    normal = perhitungan_triangle(x,10,11,12)
    rendah = perhitungan_trapezoidal(x,8,8,9,10)

    dict_al = {
        "tinggi" : tinggi,
        "sedang" : normal,
        "rendah" : rendah
    }
    
    return dict_al

In [50]:
#fuzzifikasi
fixed_acidity = nilai_fa(fa)
residual_sugar = nilai_rs(rs)
densitas = nilai_den(d)
ph = nilai_ph(p)
sulphates = nilai_sul(s)
alkohol = nilai_al(a)

print(fixed_acidity)
print(residual_sugar)
print(densitas)
print(ph)
print(sulphates)
print(alkohol)

{'tinggi': 0, 'sedang': 1, 'rendah': 0}
{'tinggi': 0, 'sedang': 0, 'rendah': 1}
{'tinggi': 0, 'normal': 0, 'rendah': 1}
{'basa': 0, 'normal': 1, 'asam': 0}
{'tinggi': 0, 'sedang': 0, 'rendah': 1}
{'tinggi': 0, 'sedang': np.float64(0.5999999999999996), 'rendah': 0}


In [51]:
#aturan fuzzy (inference)
#aturan kualitas buruk 
#1. jika alkohol rendah dan sulphates rendah maka kualitas buruk
r1 = min(alkohol["rendah"], sulphates["rendah"])

#2. jika densitas tinggi dan alkohol rendah maka kualitas buruk
r2 = min(densitas["tinggi"], alkohol["rendah"])

#3. jika ph asam dan alkohol rendah maka kualitas buruk
r3 = min(ph["asam"], alkohol["rendah"])

#4. jika fixed acidity tinggi dan sulphates rendah maka kualitas buruk
r4 = min(fixed_acidity["tinggi"], sulphates["rendah"])

#5. jika alkohol dan sulphates rendah dan densitas tinggi maka kualitas buruk
r5 = min(alkohol["rendah"], sulphates["rendah"], densitas["tinggi"])

#aturan kualitas standar
#1. jika ph normal dan residual sugar sedang dan alkohol sedang maka kualitas standar
r6 = min(ph["normal"], residual_sugar["sedang"], alkohol["sedang"])

#2. jika alkohol sedang dan densitas normal maka kualitas standar
r7 = min(alkohol["sedang"], densitas["normal"])

#3. jika ph normal dan sulphates sedang
r8 = min(ph["normal"], sulphates["sedang"])

#4. jika residual sugar dan alkohol sedang
r9 = min( residual_sugar["sedang"], alkohol["sedang"])

#5. jika fixed acidity sedang
r10 = fixed_acidity["sedang"]

#aturan jika kualitas tinggi
#1. jika alkohol tinggi dan sulphates tinggi
r11 = min(alkohol["tinggi"], sulphates["tinggi"])

#2. jika densitas rendah dan alkohol tinggi
r12 = min(alkohol["tinggi"], densitas["rendah"])

#3. jika ph normal dan alkohol tinggi
r13 = min(alkohol["tinggi"], ph["normal"])

#4. jika residual tinggi dan sulphates tinggi
r14 = min(residual_sugar["tinggi"], sulphates["tinggi"])

#5. jika alkohol tinggi dan densitas rendah dan sulphates sedang atau tinggi
r15 = min(densitas["rendah"], (max(densitas["normal"],densitas["tinggi"])), alkohol["tinggi"])

buruk = max(r1,r2,r3,r4,r5)
standar = max(r5,r6,r7,r8,r9,r10)
bagus = max(r11,r12,r13,r14,r15)

print(buruk, standar, bagus)

0 1 0


In [66]:
#defuzzifikasi
def nilai_ph(x):
    tinggi = perhitungan_trapezoidal(x,70,80,100,100)
    normal = perhitungan_triangle(x,40,55,70)
    rendah = perhitungan_trapezoidal(x,10,10,30,40)
    return max(tinggi, normal, rendah)
    
z = []

for i in range(10,100,5):
    n = nilai_ph(i)
    x = n*i
    z.append(x)

print(z)


[0, 15, 20, 25, 30, 17.5, 0, 15.0, 33.33333333333333, 55, 40.0, 21.666666666666664, 0, 37.5, 80, 85, 90, 95]
